# EXP-025 | LGB 하이퍼파라미터 재튜닝 (TE 피처 포함)

EXP-019 LGB 튜닝은 TE 없는 피처셋 기준. TE 추가로 피처 수/분포가 달라졌으므로
새 피처셋에 맞는 최적 파라미터 탐색.

**효율화**: TE를 Optuna 전에 한 번만 계산 (OOF 방식) → 50 trials 전체에서 재사용.

| 항목 | 내용 |
|---|---|
| 피처 | FE-v2 + 단일 TE 9개 (EXP-023 동일) |
| Optuna | 50 trials, TPESampler |
| 출력 | 최적 LGB 파라미터 (다음 앙상블 실험에 사용) |
| 기준선 | EXP-023 LGB OOF = 확인 후 비교 |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 25
AUTHOR   = '조여진'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비 (FE v2)

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']     = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']       = df[male_cols].sum(axis=1)
    df['여성_불임_합계']       = df[female_cols].sum(axis=1)
    df['부부_불임_합계']       = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']       = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']       = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']     = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']   = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    df['임신_성공률']   = df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)
    df['시술_실패_횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    egg_count = df['수집된 신선 난자 수'].fillna(-1)
    df['최적_난자수']    = ((egg_count >= 5) & (egg_count <= 15)).astype(int)
    df['나이_이식배아수'] = df['시술 당시 나이'] * df['이식된 배아 수'].fillna(0)
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 89)  /  X_test: (90067, 89)


## 2. TE 사전 계산 (Optuna 전 1회만)

OOF 방식으로 train TE 계산 → Optuna 50 trials 전체에서 재사용해 속도 확보.

In [3]:
TE_COLS = [
    '시술 시기 코드', '시술 유형', '특정 시술 유형',
    '배란 유도 유형', '난자 출처', '정자 출처',
    '시술 당시 나이', '총 시술 횟수', '총 임신 횟수',
]
TE_COLS  = [c for c in TE_COLS if c in X_train.columns]
TE_ALPHA = 10

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# train: OOF TE (val 행은 해당 fold의 tr 통계로 계산)
X_train_te = X_train.copy()
for col in TE_COLS:
    X_train_te[f'{col}_te'] = np.nan

for tr_idx, val_idx in skf.split(X_train, y_train):
    X_tr = X_train.iloc[tr_idx]
    y_tr = y_train.iloc[tr_idx]
    gm   = y_tr.mean()
    for col in TE_COLS:
        te_col   = f'{col}_te'
        grp      = y_tr.groupby(X_tr[col])
        cnt      = grp.count()
        mean_t   = grp.mean()
        smoothed = (cnt * mean_t + TE_ALPHA * gm) / (cnt + TE_ALPHA)
        vals = X_train.iloc[val_idx][col].map(smoothed).fillna(gm).values
        X_train_te.iloc[val_idx, X_train_te.columns.get_loc(te_col)] = vals

# NaN 잔존 시 전역 평균으로 보정
gm_global = y_train.mean()
for col in TE_COLS:
    X_train_te[f'{col}_te'] = X_train_te[f'{col}_te'].fillna(gm_global)

print(f'X_train_te: {X_train_te.shape}')
print(f'TE 컬럼 ({len(TE_COLS)}개): {", ".join(TE_COLS)}')

X_train_te: (256351, 98)
TE 컬럼 (9개): 시술 시기 코드, 시술 유형, 특정 시술 유형, 배란 유도 유형, 난자 출처, 정자 출처, 시술 당시 나이, 총 시술 횟수, 총 임신 횟수


## 3. Optuna LGB 튜닝

In [4]:
def objective(trial):
    params = dict(
        objective='binary', metric='auc', verbosity=-1, seed=SEED,
        num_threads=-1, is_unbalance=True, bagging_freq=1,
        learning_rate      = trial.suggest_float('learning_rate',       0.01,  0.1,  log=True),
        num_leaves         = trial.suggest_int(  'num_leaves',          20,    400),
        max_depth          = trial.suggest_int(  'max_depth',           3,     8),
        min_child_samples  = trial.suggest_int(  'min_child_samples',   10,    300),
        feature_fraction   = trial.suggest_float('feature_fraction',    0.3,   1.0),
        bagging_fraction   = trial.suggest_float('bagging_fraction',    0.5,   1.0),
        lambda_l1          = trial.suggest_float('lambda_l1',           1e-3,  15.0, log=True),
        lambda_l2          = trial.suggest_float('lambda_l2',           1e-3,  15.0, log=True),
        min_gain_to_split  = trial.suggest_float('min_gain_to_split',   0.0,   1.0),
    )
    oof = np.zeros(len(X_train_te))
    for tr_idx, val_idx in skf.split(X_train_te, y_train):
        X_tr  = X_train_te.iloc[tr_idx]
        X_val = X_train_te.iloc[val_idx]
        y_tr  = y_train.iloc[tr_idx]
        y_val = y_train.iloc[val_idx]
        ds_tr  = lgb.Dataset(X_tr,  label=y_tr)
        ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
        m = lgb.train(params, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                      callbacks=[lgb.early_stopping(100, verbose=False),
                                 lgb.log_evaluation(-1)])
        oof[val_idx] = m.predict(X_val)
    return roc_auc_score(y_train, oof)

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n최적 OOF AUC : {study.best_value:.5f}')
print(f'최적 파라미터:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

  0%|          | 0/50 [00:00<?, ?it/s]


최적 OOF AUC : 0.74051
최적 파라미터:
  learning_rate: 0.01369375634677629
  num_leaves: 126
  max_depth: 5
  min_child_samples: 182
  feature_fraction: 0.3284851940428519
  bagging_fraction: 0.6459035591898032
  lambda_l1: 1.2016221908502045
  lambda_l2: 0.02431385472202543
  min_gain_to_split: 0.3216337223328029


## 4. 결과 정리 & 다음 실험용 파라미터 출력

In [5]:
bp = study.best_params

print('=' * 60)
print(f'EXP-025 최적 LGB (TE 포함 피처셋)')
print(f'OOF AUC: {study.best_value:.5f}')
print('=' * 60)
print()
print('# ── 다음 앙상블 노트북에 복사 ──────────────────────────')
print('LGB_PARAMS = dict(')
print("    objective='binary', metric='auc', verbosity=-1, seed=SEED,")
print("    num_threads=-1, is_unbalance=True, bagging_freq=1,")
print(f"    learning_rate={bp['learning_rate']},")
print(f"    num_leaves={bp['num_leaves']},")
print(f"    max_depth={bp['max_depth']},")
print(f"    min_child_samples={bp['min_child_samples']},")
print(f"    feature_fraction={bp['feature_fraction']},")
print(f"    bagging_fraction={bp['bagging_fraction']},")
print(f"    lambda_l1={bp['lambda_l1']},")
print(f"    lambda_l2={bp['lambda_l2']},")
print(f"    min_gain_to_split={bp['min_gain_to_split']},")
print(')')

# 상위 10 trial 확인
print()
print('상위 10 trials:')
trials_df = study.trials_dataframe().sort_values('value', ascending=False)
print(trials_df[['number', 'value']].head(10).to_string(index=False))

EXP-025 최적 LGB (TE 포함 피처셋)
OOF AUC: 0.74051

# ── 다음 앙상블 노트북에 복사 ──────────────────────────
LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.01369375634677629,
    num_leaves=126,
    max_depth=5,
    min_child_samples=182,
    feature_fraction=0.3284851940428519,
    bagging_fraction=0.6459035591898032,
    lambda_l1=1.2016221908502045,
    lambda_l2=0.02431385472202543,
    min_gain_to_split=0.3216337223328029,
)

상위 10 trials:
 number    value
     45 0.740511
     24 0.740510
     42 0.740501
     26 0.740495
     29 0.740478
     33 0.740475
     40 0.740474
     49 0.740468
     37 0.740459
     32 0.740450


In [6]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score

# 최적 파라미터로 최종 OOF 재계산 (leaderboard용)
final_params = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    **study.best_params
)
oof_final = np.zeros(len(X_train_te))
for tr_idx, val_idx in skf.split(X_train_te, y_train):
    ds_tr  = lgb.Dataset(X_train_te.iloc[tr_idx],  label=y_train.iloc[tr_idx])
    ds_val = lgb.Dataset(X_train_te.iloc[val_idx],  label=y_train.iloc[val_idx], reference=ds_tr)
    m = lgb.train(final_params, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    oof_final[val_idx] = m.predict(X_train_te.iloc[val_idx])

oof_binary = (oof_final >= 0.5).astype(int)
best_auc   = roc_auc_score(y_train, oof_final)

params_str = (f'LGB Optuna 50trials | FE-v2+TE({len(TE_COLS)}cols) | '
              f'lr={bp["learning_rate"]:.4f} leaves={bp["num_leaves"]} depth={bp["max_depth"]}')
NOTES    = 'EXP-019 이후 TE 피처 추가 상태에서 LGB 재튜닝'
INSIGHTS = f'EXP-019 대비 변화 확인 | 피처 수 {X_train_te.shape[1]}'

log_to_leaderboard(
    EXP_NO, AUTHOR, 'LGB-Optuna-v2(TE)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    best_auc, f'Stratified {N_FOLDS}-Fold', 'v2+TE', X_train_te.shape[1],
    'is_unbalance', 'N', None,
    'notebooks/21_hparam_lgb_te_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-025 기록 완료 (row 23)
